# Outliers and val error

Looking at which outliers make the val score so high, and if removing them from fit is really a good idea.

In [ ]:
from prc_fuel.clean import clean, ground_outliers, rate_outliers
from prc_fuel.evaluate import rmse, score
from prc_fuel.features import FEATURES, build_features
from prc_fuel.load import load
from prc_fuel.split import split
from prc_fuel.train import train

In [ ]:
features = list(FEATURES)


df = load("train")
fit, val = split(df)
fit = clean(fit)
model = train(fit, features)


X_val = build_features(val, features)
y_pred = model.predict(X_val)
val = val.assign(pred=y_pred)

rmse(val.fuel_kg, val.pred)

297.6, same as run.py so it's the same model.

In [ ]:
rate = rate_outliers(val)
ground = ground_outliers(val)

val = val.assign(group="normal")
val.loc[val.flight_id.isin(rate), "group"] = "rate"
val.loc[val.flight_id.isin(ground), "group"] = "ground"

val.groupby("group").flight_id.nunique()

77 outlier flights out of 2207 in val. 42 ground, 35 rate (flights that are both end up in ground).

In [ ]:
val = val.assign(sq_err=(val.fuel_kg - val.pred) ** 2)
by_group = val.groupby("group").sq_err.sum()
by_group / by_group.sum()

rate flights = 73% of the error, ground ~3.5%, normal flights 23%. So it's mostly the rate ones, and we can't detect those without fuel_kg, so not much to do about them.

In [ ]:
fit_raw, _ = split(df)
rate_fit = rate_outliers(fit_raw)
ground_fit = ground_outliers(fit_raw)
rank = load("rank")

variants = {
    "none": set(),
    "rate": rate_fit,
    "ground": ground_fit,
    "all": rate_fit | ground_fit,
}

for name, bad in variants.items():
    fit_v = fit_raw[~fit_raw.flight_id.isin(bad)]
    model_v = train(fit_v, features)
    print(name, score(model_v, val, features), score(model_v, rank, features))

Removing rate from fit makes rank worse (248 -> 262 with nothing vs rate only). The training mean goes from ~561 to ~555 without them, they're mostly flights that burn a lot, so the model ends up predicting a bit too low.
ground only is best on val (293.8) and rank is close to none (250.7 vs 248). Keeping that one. Val differences are small though (<4 kg).